# 00 Projektumfang und Anforderungen

## Zweck
Definieren Sie das Luftqualitätsprojekt, die Leitfrage, die offiziellen Anforderungen, die Nur-Notebook-Entscheidung und Nichtziele.

## Eingaben
Projektanforderungen, freigegebene Quellenliste und Repository-Dokumentation.

## Ausgaben
Eine gepruefte Ausfuehrungsumgebung, eine klare Anforderungs-zu-Notebook-Karte und Pipeline-Uebersicht.


## Verwendete Technologien
Markdown, Mermaid, Jupyter.

## Konfiguration und Infrastrukturstart

Dieses erste Notebook waehlt die Ausfuehrungsumgebung und prueft die Infrastruktur. Es sind keine externen Hilfsskripte erforderlich.

- `PROJECT_EXECUTION_MODE=auto` bevorzugt Docker Desktop und verwendet andernfalls eine konfigurierte FH-`.env`.
- `PROJECT_EXECUTION_MODE=docker` erzwingt den lokalen Docker-Desktop-Weg.
- `PROJECT_EXECUTION_MODE=fh` erzwingt die FH-Umgebung.

Beim ersten lokalen Docker-Start wird der Compose-Stack aufgebaut. Danach dieses Notebook im gestarteten Jupyter unter `http://localhost:8888` erneut oeffnen und ausfuehren. Die Notebooks `01` bis `09` werden dort in Reihenfolge ausgefaehrt.


### Projektpfade auflösen

Die erste Codezelle bestimmt das Repository-Stammverzeichnis und leitet die lokalen `data/`- und Checkpoint-Ordner ab. Dieses Muster wiederholt sich in ausführbaren Notebooks, sodass sie vom Repository-Stammverzeichnis und vom Ordner `notebooks/` aus funktionieren.

In [ ]:
from pathlib import Path
import os

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))

### Ausfuehrungsumgebung auswaehlen und pruefen

Die folgende Zelle startet bei Bedarf Docker Desktop und Docker Compose direkt aus dem Notebook. Innerhalb des Docker-Jupyter-Containers prueft sie Kafka und Spark ueber das interne Compose-Netzwerk. Fuer die FH-Umgebung liest sie die nicht versionierte `.env` und lehnt Platzhalter kontrolliert ab.


In [ ]:
from __future__ import annotations

from dotenv import dotenv_values, load_dotenv
import platform
import shutil
import socket
import subprocess
import time

load_dotenv(PROJECT_ROOT / ".env", override=False)

REQUESTED_EXECUTION_MODE = os.getenv("PROJECT_EXECUTION_MODE", "auto").lower()
assert REQUESTED_EXECUTION_MODE in {"auto", "docker", "fh"}, (
    "PROJECT_EXECUTION_MODE muss auto, docker oder fh sein."
)


def tcp_check(endpoint: str, timeout_seconds: float = 3.0) -> dict:
    if ":" not in endpoint:
        return {"endpoint": endpoint, "reachable": False, "reason": "kein Host-Port-Endpunkt"}
    host, port_text = endpoint.rsplit(":", 1)
    try:
        with socket.create_connection((host, int(port_text)), timeout=timeout_seconds):
            return {"endpoint": endpoint, "reachable": True, "reason": None}
    except (OSError, ValueError) as exc:
        return {"endpoint": endpoint, "reachable": False, "reason": str(exc)}


def wait_for_endpoints(endpoints: dict[str, str], timeout_seconds: int = 180) -> dict[str, dict]:
    deadline = time.monotonic() + timeout_seconds
    latest = {}
    while time.monotonic() < deadline:
        latest = {name: tcp_check(endpoint) for name, endpoint in endpoints.items()}
        if all(result["reachable"] for result in latest.values()):
            return latest
        time.sleep(3)
    return latest


def docker_engine_ready() -> bool:
    if not shutil.which("docker"):
        return False
    result = subprocess.run(["docker", "info"], capture_output=True, text=True)
    return result.returncode == 0


def start_docker_desktop_if_needed() -> None:
    if docker_engine_ready():
        return
    if platform.system() == "Windows":
        docker_desktop = Path(os.environ.get("ProgramFiles", r"C:\Program Files")) / "Docker" / "Docker" / "Docker Desktop.exe"
        if docker_desktop.exists():
            subprocess.Popen([str(docker_desktop)])
            deadline = time.monotonic() + 180
            while time.monotonic() < deadline:
                if docker_engine_ready():
                    return
                time.sleep(3)
    raise RuntimeError("Docker Desktop ist nicht gestartet oder die Docker-CLI ist nicht verfuegbar.")


def verify_fh_environment() -> dict:
    config = {key: value for key, value in dotenv_values(PROJECT_ROOT / ".env").items()}
    required = ["SPARK_MASTER_URL", "KAFKA_BOOTSTRAP_SERVERS", "KAFKA_TOPIC_AIR_QUALITY_LIVE"]
    invalid = [key for key in required if not config.get(key) or "<" in config[key] or "gXX" in config[key]]
    assert not invalid, f"FH-.env enthaelt fehlende Werte oder Platzhalter: {invalid}"
    assert config["SPARK_MASTER_URL"].startswith("spark://"), "FH-SPARK_MASTER_URL muss mit spark:// beginnen."
    checks = {
        "kafka": tcp_check(config["KAFKA_BOOTSTRAP_SERVERS"]),
        "spark_master": tcp_check(config["SPARK_MASTER_URL"].removeprefix("spark://")),
    }
    assert all(result["reachable"] for result in checks.values()), f"FH-Infrastruktur nicht erreichbar: {checks}"
    return {"selected_environment": "fh", "checks": checks, "config_source": str(PROJECT_ROOT / ".env")}


inside_docker_jupyter = Path("/.dockerenv").exists() and os.getenv("EXECUTION_ENV", "").startswith("docker_")

if inside_docker_jupyter:
    docker_checks = {
        "kafka": tcp_check("kafka:29092"),
        "spark_master": tcp_check("spark-master:7077"),
    }
    assert all(result["reachable"] for result in docker_checks.values()), f"Docker-Infrastruktur nicht erreichbar: {docker_checks}"
    infrastructure_status = {
        "selected_environment": "docker",
        "runtime": "docker_jupyter",
        "checks": docker_checks,
        "next_step": "Notebooks 01 bis 09 in diesem Jupyter ausfuehren.",
    }
elif REQUESTED_EXECUTION_MODE in {"auto", "docker"}:
    try:
        start_docker_desktop_if_needed()
        subprocess.run(["docker", "compose", "up", "--build", "-d"], cwd=PROJECT_ROOT, check=True)
        host_checks = wait_for_endpoints({
            "kafka": "localhost:9092",
            "spark_master": "localhost:7077",
            "jupyter": "localhost:8888",
        })
        assert all(result["reachable"] for result in host_checks.values()), f"Docker-Dienste nicht erreichbar: {host_checks}"
        infrastructure_status = {
            "selected_environment": "docker",
            "runtime": "local_bootstrap",
            "checks": host_checks,
            "next_step": "http://localhost:8888 oeffnen, Notebook 00 dort erneut ausfuehren und anschliessend mit Notebook 01 fortfahren.",
        }
    except Exception:
        if REQUESTED_EXECUTION_MODE == "docker":
            raise
        infrastructure_status = verify_fh_environment()
else:
    infrastructure_status = verify_fh_environment()

print(infrastructure_status)


## Umsetzung
Das Projekt verwendet EEA, Wikipedia und Open-Meteo, um PM2.5, PM10 und NO2 in ausgewählten europäischen Städten zu untersuchen. Die Umsetzung erfolgt in Notebooks und wird über ein öffentliches Repository geteilt. Nicht vorgesehen sind Dashboard, ML-Modell, Airflow, dbt, PostgreSQL als Kernkomponente und Cloud-Bereitstellung.

In [ ]:
requirements = {'file_source': '03', 'web_scraping': '04', 'rest_api': '05', 'kafka': '05', 'spark_streaming': '06', 'parquet': '02-07', 'storytelling': '08'}
requirements

## Validierung und Qualitätsprüfung
Prüfen, ob jede Projektanforderung mindestens einem Notebook zugeordnet ist.

In [ ]:
REQUIRED_KEYS = {'file_source', 'web_scraping', 'rest_api', 'kafka', 'spark_streaming', 'parquet', 'storytelling'}
missing_requirements = REQUIRED_KEYS - set(requirements)
assert not missing_requirements, f"Nicht zugeordnete Projektanforderungen: {missing_requirements}"
print('Zuordnung der Anforderungen vollständig')

## Ergebnisse
Die Notebook-Reihenfolge gibt den Projektausführungspfad an.

## Einschr?nkungen
Dieses Notebook startet oder pr?ft ausschlie?lich die Infrastruktur. Die fachliche Datenverarbeitung beginnt in Notebook `01`.

## Nächster Schritt
Führen Sie das Notebook `01` aus, um die Machbarkeit der Quelle und Clusterergebnisse zu dokumentieren.